In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.table import Table, hstack, vstack
from sklearn.neighbors import KernelDensity
from tqdm import tqdm
import astropy.units as u
from astropy.coordinates import SkyCoord
import mpl_scatter_density

import fitsio
import numpy as np
from matplotlib import cm
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap


```sql
SELECT
    main.object_id,
    main.ra,
    main.dec,

    ------- flux and flux errors -------
    main2.g_psfflux_flux, main2.r_psfflux_flux, main2.i_psfflux_flux, main2.z_psfflux_flux, main2.y_psfflux_flux,
    main2.g_psfflux_fluxerr, main2.r_psfflux_fluxerr, main2.i_psfflux_fluxerr, main2.z_psfflux_fluxerr, main2.y_psfflux_fluxerr,
    main.g_cmodel_flux, main.r_cmodel_flux, main.i_cmodel_flux, main.z_cmodel_flux, main.y_cmodel_flux,
    main.g_cmodel_fluxerr, main.r_cmodel_fluxerr, main.i_cmodel_fluxerr, main.z_cmodel_fluxerr, main.y_cmodel_fluxerr,
------- Fiber flux and flux errors -------
main4.g_convolvedflux_2_15_flux as g_fiber_flux, main4.r_convolvedflux_2_15_flux as r_fiber_flux,
main4.i_convolvedflux_2_15_flux as i_fiber_flux, main4.z_convolvedflux_2_15_flux as z_fiber_flux, main4.y_convolvedflux_2_15_flux as y_fiber_flux,
main4.g_convolvedflux_2_15_fluxerr as g_fiber_fluxerr, main4.r_convolvedflux_2_15_fluxerr as r_fiber_fluxerr, main4.i_convolvedflux_2_15_fluxerr as i_fiber_fluxerr,
main4.z_convolvedflux_2_15_fluxerr as z_fiber_fluxerr, main4.y_convolvedflux_2_15_fluxerr as y_fiber_fluxerr,

main5.g_undeblended_convolvedflux_2_15_flux as g_fiber_tot_flux, main5.r_undeblended_convolvedflux_2_15_flux as r_fiber_tot_flux,
main5.i_undeblended_convolvedflux_2_15_flux as i_fiber_tot_flux, main5.z_undeblended_convolvedflux_2_15_flux as z_fiber_tot_flux, main5.y_undeblended_convolvedflux_2_15_flux as y_fiber_tot_flux,
main5.g_undeblended_convolvedflux_2_15_fluxerr as g_fiber_tot_fluxerr, main5.r_undeblended_convolvedflux_2_15_fluxerr as r_fiber_tot_fluxerr, main5.i_undeblended_convolvedflux_2_15_fluxerr as i_fiber_tot_fluxerr,
main5.z_undeblended_convolvedflux_2_15_fluxerr as z_fiber_tot_fluxerr, main5.y_undeblended_convolvedflux_2_15_fluxerr as y_fiber_tot_fluxerr,
    ------- extinction -------
    main.a_g, main.a_r, main.a_i, main.a_z, main.a_y,

    ------- fraction of flux in de Vaucouleur component -------
    main.g_cmodel_fracdev, main.r_cmodel_fracdev, main.i_cmodel_fracdev, main.z_cmodel_fracdev,
     ------- shape measurements -------
    main.g_extendedness_value, main.r_extendedness_value, main.i_extendedness_value, main.z_extendedness_value,
    main.g_extendedness_flag, main.r_extendedness_flag, main.i_extendedness_flag, main.z_extendedness_flag,
    main2.i_sdssshape_shape11, main2.i_sdssshape_shape22, main2.i_sdssshape_shape12,
    main2.i_sdssshape_shape11err, main2.i_sdssshape_shape22err, main2.i_sdssshape_shape12err,
    ------- flags -------
    main2.g_sdsscentroid_flag, main2.r_sdsscentroid_flag, main2.i_sdsscentroid_flag, main2.z_sdsscentroid_flag, main2.y_sdsscentroid_flag, 
    main.g_pixelflags_edge, main.r_pixelflags_edge, main.i_pixelflags_edge, main.z_pixelflags_edge, main.y_pixelflags_edge, 
    main.g_pixelflags_interpolatedcenter, main.r_pixelflags_interpolatedcenter, main.i_pixelflags_interpolatedcenter, main.z_pixelflags_interpolatedcenter, main.y_pixelflags_interpolatedcenter, 
    main.g_pixelflags_saturatedcenter, main.r_pixelflags_saturatedcenter, main.i_pixelflags_saturatedcenter, main.z_pixelflags_saturatedcenter, main.y_pixelflags_saturatedcenter, 
    main.g_pixelflags_crcenter, main.r_pixelflags_crcenter, main.i_pixelflags_crcenter, main.z_pixelflags_crcenter, main.y_pixelflags_crcenter, 
    main.g_pixelflags_bad, main.r_pixelflags_bad, main.i_pixelflags_bad, main.z_pixelflags_bad, main.y_pixelflags_bad, 
    main.g_cmodel_flag, main.r_cmodel_flag, main.i_cmodel_flag, main.z_cmodel_flag, main.y_cmodel_flag,
    -----Star masks -----
    mask.g_mask_brightstar_any, mask.g_mask_brightstar_halo, mask.g_mask_brightstar_dip,
    mask.g_mask_brightstar_ghost, mask.g_mask_brightstar_blooming, mask.g_mask_brightstar_ghost12, mask.g_mask_brightstar_ghost15,
    mask.r_mask_brightstar_any, mask.r_mask_brightstar_halo, mask.r_mask_brightstar_dip,
    mask.r_mask_brightstar_ghost, mask.r_mask_brightstar_blooming, mask.r_mask_brightstar_ghost12, mask.r_mask_brightstar_ghost15,
    mask.i_mask_brightstar_any, mask.i_mask_brightstar_halo, mask.i_mask_brightstar_dip,
    mask.i_mask_brightstar_ghost, mask.i_mask_brightstar_blooming, mask.i_mask_brightstar_ghost12, mask.i_mask_brightstar_ghost15,
    mask.z_mask_brightstar_any, mask.z_mask_brightstar_halo, mask.z_mask_brightstar_dip,
    mask.z_mask_brightstar_ghost, mask.z_mask_brightstar_blooming, mask.z_mask_brightstar_ghost12, mask.z_mask_brightstar_ghost15,
    mask.y_mask_brightstar_any, mask.y_mask_brightstar_halo, mask.y_mask_brightstar_dip,
    mask.y_mask_brightstar_ghost, mask.y_mask_brightstar_blooming, mask.y_mask_brightstar_ghost12, mask.y_mask_brightstar_ghost15

FROM
    pdr3_wide.forced main
    LEFT JOIN pdr3_wide.forced2 main2 USING (object_id)
    LEFT JOIN pdr3_wide.forced4 main4 USING (object_id)
    LEFT JOIN pdr3_wide.forced5 main5 USING (object_id)
	LEFT JOIN pdr3_wide.masks mask USING (object_id)

WHERE
    isprimary
    AND i_cmodel_mag<25.6
    ---AND boxSearch(coord, 33.5, 37.5, -7.0, -3.0) ---DESI-XMM
    ---AND boxSearch(coord, 148, 152, 4, 0) ---DESI-COSMOS
    ---AND boxSearch(coord, 149, 151, 3, 1) ---DESI-COSMOS-trimmed-for-PFS
    AND boxSearch(coord, 34.5, 36.5, -6.0, -4.0) ---DESI-XMM-trimmed-for-PFS
    ---AND boxSearch(coord, 213, 217, 54, 51.75) ---DEEP23
    ---AND boxSearch(coord, 243, 249, 41, 45) ---HERCULES

```

In [ ]:
params = {
    "legend.fontsize": "x-large",
    "axes.labelsize": "x-large",
    "axes.titlesize": "x-large",
    "xtick.labelsize": "x-large",
    "ytick.labelsize": "x-large",
    "figure.facecolor": "w",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif"
}
plt.rcParams.update(params)

def cmap_white(cmap_name):
    """Returns a colormap with white as the lowest value color."""
    import numpy as np
    try:
        from matplotlib import cm
        from matplotlib.colors import ListedColormap
        cmap = cm.get_cmap(cmap_name, 256)
    except ValueError:
        import seaborn as sns
        cmap = sns.color_palette("flare", as_cmap=True)
    newcolors = cmap(np.linspace(0, 1, 256))
    white = np.array([1, 1, 1, 0])
    newcolors[:1, :] = white
    cmap_white = ListedColormap(newcolors)
    return cmap_white

In [ ]:
base_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/")
patch = "PFS-XMM"
hsc_path = base_path / "target_data"/ f"HSC_{patch}_I_mag_lim_25.6.fits"

In [ ]:
def flux_to_mag(flux):
    return -2.5*np.log10(flux*1e-9) + 8.90

In [ ]:
hsc_cat = Table.read(hsc_path).to_pandas()
hsc_cat["i_mag"] = flux_to_mag(hsc_cat["i_cmodel_flux"])-hsc_cat["a_i"]
hsc_cat["r_mag"] = flux_to_mag(hsc_cat["r_cmodel_flux"])-hsc_cat["a_r"]
hsc_cat["z_mag"] = flux_to_mag(hsc_cat["z_cmodel_flux"])-hsc_cat["a_z"]
hsc_cat["g_mag"] = flux_to_mag(hsc_cat["g_cmodel_flux"])-hsc_cat["a_g"]

hsc_cat["g_fiber_mag"] = flux_to_mag(hsc_cat["g_fiber_flux"])-hsc_cat["a_g"]
hsc_cat["i_fiber_mag"] = flux_to_mag(hsc_cat["i_fiber_flux"])-hsc_cat["a_i"]
hsc_cat["r_fiber_mag"] = flux_to_mag(hsc_cat["r_fiber_flux"])-hsc_cat["a_r"]
hsc_cat["z_fiber_mag"] = flux_to_mag(hsc_cat["z_fiber_flux"])-hsc_cat["a_z"]

hsc_cat["i_mag_psf"] = flux_to_mag(hsc_cat["i_psfflux_flux"])-hsc_cat["a_i"]
# hsc_cat["i_fiber_tot_mag"] = flux_to_mag(hsc_cat["i_fiber_tot_flux"])-hsc_cat["a_i"]

In [ ]:
## Quality cuts
# valid I-band flux
qmask = np.isfinite(hsc_cat["i_cmodel_flux"]) & (hsc_cat["i_cmodel_flux"]>0)
#cmodel fit not failed
qmask &= (~hsc_cat["i_cmodel_flag"].values)
#General Failure Flag
qmask &= (~hsc_cat["i_sdsscentroid_flag"].values)


# Possible cuts: Bright objects nearby, bad pixels

#star-galaxy separation (is point source in I band)
extendedness = hsc_cat["i_mag_psf"]-hsc_cat["i_mag"]
mask_ext = (hsc_cat["i_mag"]>23) | (extendedness>0.02)

#bright-star-mask
bright_mask = hsc_cat["i_mask_brightstar_any"]

i_min = 20 #potentially 20
i_max = 25.3
i_mask = (hsc_cat["i_mag"] <i_max) & (hsc_cat["i_mag"] >i_min)

#We have decided to not have any color cut or fiber magnitude cut
# i_fiber_min = 18
# i_fiber_max = 25 
# mask &= (hsc_cat["i_fiber_mag"] <i_fiber_max) & (hsc_cat["i_fiber_mag"] >i_fiber_min)

plot i-mage vs i-fiber-mag distribution for the parent sample

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5,5),subplot_kw={"projection":"scatter_density"})
ax.scatter_density(hsc_cat[qmask]["i_mag"], hsc_cat[qmask]["i_fiber_mag"],cmap=cmap_white("viridis"),dpi=1000)
ax.set_aspect('equal')
x = np.linspace(15,25,100)
ax.plot(x,x, ls="--", c="k", alpha=0.5)
ax.set_xlim(16,26)
ax.set_ylim((16,26))
ax.set_xlabel("$i$ mag")
ax.set_ylabel("$i$ fiber mag")

Plot i-mag distribution of various other DESI samples

In [ ]:
for i in np.linspace(14,24.5,22):
    plt.axvline(i,c="k",ls="--",alpha=0.2,lw=1)

plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["r_mag"]<19.5)],bins=50, histtype="step", label = "~BGS")
plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["r_mag"]<20.1)],bins=50, histtype="step", label = "~BGS Faint")
plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["r_mag"]<21)],bins=50, histtype="step", label = "~BGS Fainter")
plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["z_mag"]<20.8)],bins=50, histtype="step", label = "~DC3R2")
plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["z_mag"]<21.25) & (hsc_cat["z_mag"]<22)],bins=50, histtype="step", label = "~4C3R2")
plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["z_mag"]<21)],bins=50, histtype="step", label = "~LRG")

plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["g_mag"]<23.4)],bins=50, histtype="step", label = "~ELG")
plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["i_mag"]<24.1)],bins=50, histtype="step", label="LSST Y1")
plt.hist(hsc_cat["i_mag"][qmask & (hsc_cat["i_mag"]<24.5)],bins=50, histtype="step", label="Proposed",color="k")

    
plt.xlabel("$i$-mag", fontsize=20)
plt.yscale("log")
plt.legend(loc="upper left",fontsize=10)


Star-Galaxy separation

In [ ]:
extendedness = hsc_cat["i_mag_psf"]-hsc_cat["i_mag"]
mask_ext = (hsc_cat["i_mag"]>23) | (extendedness>0.02)



qmask_gr = np.isfinite(hsc_cat["g_cmodel_flux"]) & (hsc_cat["g_cmodel_flux"]>0)
qmask_gr &= np.isfinite(hsc_cat["r_cmodel_flux"]) & (hsc_cat["r_cmodel_flux"]>0)
#cmodel fit not failed
qmask_gr &= (~hsc_cat["g_cmodel_flag"].values)
qmask_gr &= (~hsc_cat["r_cmodel_flag"].values)
#General Failure Flag
qmask_gr &= (~hsc_cat["g_sdsscentroid_flag"].values)
qmask_gr &= (~hsc_cat["r_sdsscentroid_flag"].values)


sels_cat = hsc_cat[qmask & i_mask & qmask_gr & mask_ext]
sels_cat = sels_cat.reset_index()

gr = sels_cat["g_mag"] - sels_cat["r_mag"]
ri = sels_cat["r_mag"] - sels_cat["i_mag"]

value_mask = (gr>-1) & (gr<3) & (ri>-0.5) & (ri<2.5)# perform rejection sampling


fig, ax = plt.subplots(1,1,subplot_kw={"projection":"scatter_density"})
ax.scatter_density(gr[value_mask], ri[value_mask],cmap=cmap_white("viridis"),dpi=100)
ax.set_xlabel("$g-r$")
ax.set_ylabel("$r-i$")

Select the final targets and resample to a uniform distribution in i-mag

In [ ]:
sels_cat = hsc_cat[qmask & i_mask & mask_ext & bright_mask]
sels_cat = sels_cat.reset_index()

In [ ]:
def uniform_resample(data, data_min, data_max, bin_size = 0.001, seed=42, rejection_scale = None):
    #estimate the density (replace histogram by something else?)
    # data_mask = (data>data_min) & (data<=data_min) convert it into assertion instead
    data_sel = data.copy()

    bins = int((data_max-data_min)/bin_size)
    bin_edges = np.linspace(data_min,data_max,bins)
    counts, _ = np.histogram(data_sel, bins=bin_edges,density=False)
    bin_membership = np.digitize(data_sel, bin_edges,right=True)

    weights = counts[bin_membership-1]
    weights = 1/weights
    weights /= weights.sum()

    # perform rejection sampling
    if rejection_scale is None:
        rejection_scale = weights.max()
        
    rng = np.random.default_rng(seed=seed)
    sampling_mask = rng.uniform(size=len(weights)) < weights/rejection_scale
    
    return sampling_mask, weights

In [ ]:
# rejection_scales = [0*1e-4,1e-4,1e-4,1e-4,1e-4/1.3]
rejection_scales = [0*1e-4,1*1e-4,1e-4,1e-4,1e-4/1.3]
mag_bin_mins = [20,21,22,23, 24]
mag_bin_maxs = [21,22,23,24, 25.3]

In [ ]:
final_cat = []
sizes = []
weights = []
for mins,maxs,scales in zip(mag_bin_mins,mag_bin_maxs,rejection_scales):
    mag_mask = (sels_cat["i_mag"]>mins) & (sels_cat["i_mag"]<=maxs)
    sub_cat = sels_cat[mag_mask]
    sample_mask, weight = uniform_resample(sub_cat["i_mag"],mins,maxs,rejection_scale = scales)
    final_cat.append(sub_cat[sample_mask])
    weights.append(weight[sample_mask])
    sizes.append(np.sum(sample_mask))
final_cat = pd.concat(final_cat)


In [ ]:
_ = plt.hist(sels_cat["i_mag"], histtype="step",bins=100)
_ = plt.hist(final_cat["i_mag"], histtype="step",bins=100)
plt.yscale("log")
plt.ylim(-1,12000)
print(sizes)

In [ ]:
_ = plt.hist(sels_cat["i_mag"], histtype="step",bins=100,label="All Objects")
_ = plt.hist(final_cat["i_mag"], histtype="step",bins=100,label="Selected Targets")
# fba = Table(fitsio.read("../fiber_assign/saved_outputs/tertiary-targets-9999-assign.fits"))
# fba = fba[fba["NASSIGN"]>0]
# _ = plt.hist(fba["I_MAG"],bins=100,histtype="step",label="Fiber Assigned")
plt.yscale("log")
plt.ylim(50,5e4)
plt.xlabel(r"$i$-mag")
plt.ylabel("Counts")
plt.legend(frameon=False)
plt.savefig("targets.pdf",bbox_inches="tight")

In [ ]:
# rng = np.random.default_rng(seed=42)

# weights = weights/weights.sum()
# resample_idx = rng.choice(np.arange(len(sels_cat["i_mag"])), size=25000, replace=False, p=weights)
# final_sel = sels_cat.iloc[resample_idx]

# sample_weights = weights/weights.max()
# mymask = rng.uniform(size=len(sample_weights)) < sample_weights
# final_sel = sels_cat[mymask]

In [ ]:
# plt.figure(figsize=(7,5))
# # plt.hist(sels_cat["i_mag"],bins=100, histtype="step", label="Magnitude Limited Sample",lw=2)
# plt.hist(final_sel["i_mag"],bins=100, histtype="step",  label="Uniform Magnitude Sample",lw=2)

# # plt.hist(sels_cat["i_mag"][resample_idx][np.isfinite(sels_cat["specz_redshift"][resample_idx])],bins=50, histtype="step", density=True, label="Known Redshifts")
# # plt.axhline(np.mean(np.histogram(sels_cat["i_mag"][resample_idx],bins=100, density=True)[0]), c="k", ls="--")
# plt.xlabel("$i$-band Magnitude", fontsize=20)
# plt.ylabel("Normalized Frequency", fontsize=20)
# # plt.yscale("log")
# plt.legend(loc="upper left",frameon=False)
# # plt.savefig("resampled_distribution.pdf", bbox_inches="tight")

In [ ]:
# small_cat = hsc_cat.sample(n=10000)
fig, ax = plt.subplots(1,1, figsize=(10,10))
# ax.scatter(sels_cat.loc[resample_idx, "ra"],sels_cat.loc[resample_idx, "dec"],marker=".",s=1, alpha=1)
ax.scatter(final_cat["ra"],final_cat["dec"],marker=".",s=1, alpha=1)
# plt.scatter(small_cat["ra"],small_cat["dec"],marker=".",s=1)
# plt.scatter(redm["RA"],redm["DEC"],marker=".",s=5, c="r")
ra_min = 148
ra_max = 152
dec_min = 0
dec_max = 4

ax.set_xlabel("RA $\degree$", fontsize=20)
ax.set_ylabel("DEC $\degree$", fontsize=20)
ax.set_xticks(np.arange(ra_min, ra_max+1, 1))
ax.set_xticks(np.arange(ra_min, ra_max+1, 0.5), minor=True)
ax.set_yticks(np.arange(dec_min, dec_max+1, 1))
ax.set_yticks(np.arange(dec_min, dec_max+1, 0.5), minor=True)
ax.set_xlim(ra_max+0.2, ra_min-0.2)
ax.set_ylim(dec_min-0.2, dec_max+0.2)
# And a corresponding grid
ax.grid(which='both')

# Or if you want different settings for the grids:
ax.grid(which='minor', alpha=0.5)
ax.grid(which='major', alpha=0.8)
ax.set_title(f"Target Density: {len(final_cat)/16} per sq deg.")

In [ ]:
TEST_NAME = "PFS_Deep"
blank = np.zeros(len(final_cat))
Table({"RA":final_cat["ra"], "DEC":final_cat["dec"], "PMRA":blank, "PMDEC":blank,
       'REF_EPOCH':blank+2000, "OVERRIDE":(blank+1).astype(bool),"I_MAG": final_cat["i_mag"]}, ).write(base_path / f"PFS_Deep_{patch}_proposal.fits", overwrite=True)

In [ ]:
desi_data_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/pilot_obs/MERGED")
cat = Table.read(desi_data_path / "merged_cat_LSST_WL_Y1.fits")


In [ ]:
cat = cat[cat["FIELD_NAME"]=="XMMLSS"]

In [ ]:
cat = cat[['TARGETID',
'TARGET_RA',
'TARGET_DEC',
'REF_EPOCH',
'EXPTIME',
'i_cmodel_flux',
'a_i',
]]

In [ ]:
cat.write(base_path / f"DESI_DEEP_Pilot_XMM_observed.fits", overwrite=True)